In [1]:
import os
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.mask import mask
from rasterio.enums import Resampling

In [2]:
ROOT_DIR = os.path.abspath(os.getcwd())
in_path = ROOT_DIR + "\\" + 'GIS_data'

In [3]:
def upscale_raster(input_path, output_path, scale_factor):
    with rasterio.open(input_path) as src:
        new_height = int(src.height * scale_factor)
        new_width = int(src.width * scale_factor)

        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.nearest
        ).astype(float)  # convert for division

        data /= (scale_factor * scale_factor)

        new_transform = src.transform * src.transform.scale(
            (src.width / new_width),
            (src.height / new_height)
        )

        new_meta = src.meta.copy()
        new_meta.update({
            'height': new_height,
            'width': new_width,
            'transform': new_transform,
            'dtype': 'float32'  # or suitable float
        })

        with rasterio.open(output_path, 'w', **new_meta) as dst:
            dst.write(data)


### Provide link to directory containing the crop data (physical area | production)

In [4]:
# Crop Production
in_path_raster = r"GIS_data/Crop_Prod_SPAM/all_datasets/V2r0"
file = "Crop_Production"

## Physical Area
#in_path_raster = r"C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\Harvested_area\spam2020V1r0_global_physical_area.geotiff\spam2020V1r0_global_physical_area"
#file = "Physical_Area"

## Crop Production
#in_path_raster = r"C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\Harvested_area\spam2020V1r0_global_production.geotiff\spam2020V1r0_global_production"
#file = "Crop_Production"

## GLW4
# in_path_raster = r"C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\GLW4"
# file = "GLW4"

###  Provide crops you want to extract

Note: Typically there are 3 layers per crop indicating I: Irrigated, R: Rainfed, A: All (I presume). Make sure you include the prefix too e.g., RICE_A.

In [5]:
## For Crops
crop_name = ['BANA_A',
'CASS_A',
'CITR_A',
'COWP_A',
'ORTS_A',
'TROF_A',
'VEGE_A',
'POTA_A',
'SWPO_A',
'TEMF_A',
'TOMA_A']

## For animals
#crop_name = ["Cattle", "Chicken", "Buffaloes", "Pigs", "Goats", "Sheep", "Horses", "Ducks"]

In [6]:
# Read files with tif extension and assign their name into two list for discrete and continuous datasets
raster_files_dis = []
raster_files_con =[]

for i in os.listdir(in_path_raster):
    for crop in crop_name:
        if (crop in i) and i.endswith('.tif'):
            with rasterio.open(in_path_raster + '\\' + i) as src:
                data = src.read() 
                unique_val = len(np.unique(data))
                if unique_val < 20:                                   # This value is arbitrary
                    raster_files_dis.append(i)
                else:
                    raster_files_con.append(i)
                
for j in os.listdir(in_path_raster):
    if ("ncb" in j) and j.endswith('.tif'):
        with rasterio.open(in_path_raster + '\\' + j) as src:
            data = src.read() 
            unique_val = len(np.unique(data))
            if unique_val < 20:                                   # This value is arbitrary
                raster_files_dis.append(j)
            else:
                raster_files_con.append(j)
                
# keep only unique values -- Not needed but just in case there are dublicates
raster_files_con = list(set(raster_files_con))
raster_files_dis = list(set(raster_files_dis))
                
print ("We have identified {} continuous raster(s):".format(len(raster_files_con)),"\n",)
for raster in raster_files_con:
    print ( "*", raster)
    
print ("\n", "We have identified {} discrete raster(s):".format(len(raster_files_dis)),"\n",)
for raster in raster_files_dis:
    print ( "*", raster)

We have identified 11 continuous raster(s): 

* spam2020_V2r0_global_P_TEMF_A.tif
* spam2020_V2r0_global_P_CITR_A.tif
* spam2020_V2r0_global_P_BANA_A.tif
* spam2020_V2r0_global_P_SWPO_A.tif
* spam2020_V2r0_global_P_POTA_A.tif
* spam2020_V2r0_global_P_TROF_A.tif
* spam2020_V2r0_global_P_CASS_A.tif
* spam2020_V2r0_global_P_VEGE_A.tif
* spam2020_V2r0_global_P_COWP_A.tif
* spam2020_V2r0_global_P_ORTS_A.tif
* spam2020_V2r0_global_P_TOMA_A.tif

 We have identified 0 discrete raster(s): 



### Load the shapefile of the AoI you want to clip the raster layers on

In [7]:
# Define path and name of the file
admin_path = r'GIS_data/Admin/model_input/adm0/mdg_admbnda_adm0_BNGRC_OCHA_20181031.shp'

aoi = gpd.read_file(admin_path)

aoi['geometry'] = aoi.apply(lambda x:
                            x.geometry.buffer(0.08, cap_style=3), axis=1)  ### 0.008 deg = ~1km

### Clip selected raster files and save in GIS directory

In [8]:
for raster in raster_files_con:
    prefix = raster.rstrip(".tif")
    #prefix = prefix + "_" + "MDG"                     ## this might change based on what you use
    prefixres = prefix + "_R"    ## this might change based on what you use
    
    #path = r'C:\Users\alexl\Dropbox\Self-employment\SEforALL\Work\Mozambique\Cooling_Module\GIS_data\Harvested_area\spam2020V1r0_global_physical_area.geotiff\spam2020V1r0_global_physical_area'
    raster_path = in_path_raster + "//" + raster
    print (raster_path)
    
    with rasterio.open(raster_path) as src:
        # Transform the aoi geometry into the same coordinate reference system (CRS) as the raster
        aoi = aoi.to_crs(src.crs)
    
        # Extract the geometry of the aoi
        geometry = [aoi.geometry[0]]  # Assumes the shapefile has one polygon. Adjust if there are multiple geometries.
    
        # Clip the raster using the geometry
        out_image, out_transform = mask(src, geometry, crop=True)
        out_meta = src.meta.copy()
    
        # Update metadata with the new dimensions, transform, and CRS
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # Save the clipped raster
        out_path = r"GIS_data/Crop_Prod_SPAM/model_input"  ## Make sure the directory exists
        clipped_raster_path = out_path + "//" + prefix + ".tiff"
        with rasterio.open(clipped_raster_path, "w", **out_meta) as dest:
            dest.write(out_image)
            
        print("Clipping for {} completed".format(prefix))
        
        # Define the scale factor (10 km to 0.5 km requires a scale factor of 20)
        scale_factor = 20

        # Run the upscale function
        output_raster = out_path + "//" + prefixres + ".tiff"
        upscale_raster(clipped_raster_path, output_raster, scale_factor)
        
        print("Clipping for {} completed".format(prefixres))
        
        # After successful upscale
        os.remove(clipped_raster_path)


GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_TEMF_A.tif
Clipping for spam2020_V2r0_global_P_TEMF_A completed
Clipping for spam2020_V2r0_global_P_TEMF_A_R completed
GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_CITR_A.tif
Clipping for spam2020_V2r0_global_P_CITR_A completed
Clipping for spam2020_V2r0_global_P_CITR_A_R completed
GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_BANA_A.tif
Clipping for spam2020_V2r0_global_P_BANA_A completed
Clipping for spam2020_V2r0_global_P_BANA_A_R completed
GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_SWPO_A.tif
Clipping for spam2020_V2r0_global_P_SWPO_A completed
Clipping for spam2020_V2r0_global_P_SWPO_A_R completed
GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_POTA_A.tif
Clipping for spam2020_V2r0_global_P_POTA_A completed
Clipping for spam2020_V2r0_global_P_POTA_A_R completed
GIS_data/Crop_Prod_SPAM/all_datasets/V2r0//spam2020_V2r0_global_P_TROF_A.ti